
# Sprint 1 Profiling Results Summary

## Overview
Profiled four source tables for the Heavy Supplier, Inventory & Warehouse Analytics project.

- Total rows profiled: 187,692
- `sales_orders_header`: 20,000 rows
- `sales_orders_lines`: 130,402 rows
- `invoices`: 18,033 rows
- `payments`: 19,257 rows

## Key Findings
- No missing values or exact duplicate rows were detected in the four tables.
- Order headers and order lines matched, and the checked line-level calculations reconciled within a tolerance of 0.01.
- Repeated invoice IDs were identified: 18,033 invoice rows but 17,836 unique invoice IDs.
- Repeated payment IDs were identified: 19,257 payment rows but 19,055 unique payment IDs.
- Date columns were successfully checked for unparseable dates.
- Invoice and payment reconciliation requires further investigation because repeated invoice IDs may make joins ambiguous.

## Limitations and Next Steps
The findings are based on initial profiling. Repeated identifiers and invoice-payment reconciliation flags require validation before final conclusions are drawn. Raw source data has not been changed. Data cleaning and any approved corrections should follow the project's later-sprint process.


In [1]:

from pathlib import Path
import pandas as pd

# Find the four CSV files beside this notebook.
from pathlib import Path

folder = Path(r"C:\Users\Timothy\Downloads\HeavySuppliersWarehouseDatasets\HeavySuppliersWarehouseDatasets")

required_files = [
    "sales_orders_header.csv",
    "sales_orders_lines.csv",
    "invoices.csv",
    "payments.csv"
]

tables = {}

for filename in required_files:
    file_path = folder / filename

    if not file_path.exists():
        print(f"NOT FOUND: {filename}")
        print("Notebook folder:", folder)
        print("Move the notebook into the extracted CSV folder, then run again.")
    else:
        tables[filename.replace(".csv", "")] = pd.read_csv(file_path)
        print(f"Loaded: {filename}")

print("\n--- TABLE PROFILE ---")

for name, df in tables.items():
    print(f"\n{'=' * 60}")
    print("TABLE:", name)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes.to_string())

    print("\nMissing values per column:")
    print(df.isna().sum().to_string())

    print("\nExact duplicate rows:", df.duplicated().sum())

    print("\nNumeric column summary:")
    numeric_cols = df.select_dtypes(include="number").columns
    if len(numeric_cols):
        print(df[numeric_cols].describe().to_string())
    else:
        print("No numeric columns detected.")

    print("\nDate coverage:")
    date_cols = [
        col for col in df.columns
        if "date" in col.lower()
    ]

    if date_cols:
        for col in date_cols:
            dates = pd.to_datetime(df[col], errors="coerce")
            print(
                f"{col}: minimum={dates.min()}, "
                f"maximum={dates.max()}, "
                f"unparseable={dates.isna().sum()}"
            )
    else:
        print("No columns with 'date' in their names.")

print("\n--- END OF PROFILE ---")


Loaded: sales_orders_header.csv
Loaded: sales_orders_lines.csv
Loaded: invoices.csv
Loaded: payments.csv

--- TABLE PROFILE ---

TABLE: sales_orders_header
Rows: 20000
Columns: 11

Column names:
['so_id', 'customer_id', 'branch_id', 'order_date', 'delivery_date', 'order_status', 'payment_terms', 'total_order_value', 'total_gst_amount', 'grand_total', 'sales_channel']

Data types:
so_id                 object
customer_id           object
branch_id             object
order_date            object
delivery_date         object
order_status          object
payment_terms         object
total_order_value      int64
total_gst_amount     float64
grand_total          float64
sales_channel         object

Missing values per column:
so_id                0
customer_id          0
branch_id            0
order_date           0
delivery_date        0
order_status         0
payment_terms        0
total_order_value    0
total_gst_amount     0
grand_total          0
sales_channel        0

Exact duplicate 

In [2]:
print("Folder exists:", folder.exists())
print("CSV files found:")

for file in folder.glob("*.csv"):
    print(file.name)

Folder exists: True
CSV files found:
branches.csv
customers.csv
inventory_master.csv
invoices.csv
payments.csv
products.csv
purchase_orders_header.csv
purchase_orders_lines.csv
sales_orders_header.csv
sales_orders_lines.csv
stock_ledger.csv
suppliers.csv
